# Read ECOSTRESS data via NASA CMR

https://lpdaac.usgs.gov/products/eco_l2g_lstev002/

Detailed information about CMR STAC API and examples can be found at https://nasa-openscapes.github.io/2021-Cloud-Hackathon/tutorials/04_NASA_Earthdata_Authentication.html

In [1]:
from pystac_client import Client  
from collections import defaultdict    
import json
import geopandas
import geoviews as gv
from cartopy import crs
gv.extension('bokeh', 'matplotlib')

ImportError: cannot import name 'Markup' from 'jinja2' (C:\Users\ebueechi\Miniconda3\envs\yipeeo38\lib\site-packages\jinja2\__init__.py)

## Query CMR STAC API
Query the CMR STAC API to get a list of available catalogs first.

In [2]:
STAC_URL = 'https://cmr.earthdata.nasa.gov/stac'
cmr_stac = Client.open(STAC_URL)

C:\Users\ebueechi\Miniconda3\envs\yipeeo38\lib\site-packages\pystac_client\client.py:186: NoConformsTo: Server does not advertise any conformance classes.
  warnings.warn(NoConformsTo())


In [3]:
import rich.table
from rich.console import Console
console = Console()

table = rich.table.Table(title="Data Providers")
table.add_column("ID", style="cyan", no_wrap=True)
table.add_column("Provider")
  
providers = [p for p in cmr_stac.get_children()]
for count, provider in enumerate(providers):
    table.add_row(f"{count}", provider.title)

console.print(table)

  Data Providers   
┏━━━━┳━━━━━━━━━━━━┓
┃ ID ┃ Provider   ┃
┡━━━━╇━━━━━━━━━━━━┩
│ 0  │ ESA        │
│ 1  │ GHRC       │
│ 2  │ ECHO       │
│ 3  │ ISRO       │
│ 4  │ EDF_DEV04  │
│ 5  │ ASF        │
│ 6  │ EUMETSAT   │
│ 7  │ CDDIS      │
│ 8  │ JAXA       │
│ 9  │ AU_AADC    │
│ 10 │ ECHO10_OPS │
│ 11 │ LANCEAMSR2 │
│ 12 │ GESDISCCLD │
│ 13 │ GHRSSTCWIC │
│ 14 │ LARC_CLOUD │
│ 15 │ LANCEMODIS │
│ 16 │ NSIDCV0    │
│ 17 │ NSIDC_ECS  │
│ 18 │ NCCS       │
│ 19 │ OBPG       │
│ 20 │ OMINRT     │
│ 21 │ USGS_LTA   │
│ 22 │ ASIPS      │
│ 23 │ ESDIS      │
│ 24 │ NSIDC_CPRD │
│ 25 │ ORNL_CLOUD │
│ 26 │ FEDEO      │
│ 27 │ MLHUB      │
│ 28 │ LAADS      │
│ 29 │ LARC_ASDC  │
│ 30 │ LPDAAC_ECS │
│ 31 │ NOAA_NCEI  │
│ 32 │ OB_DAAC    │
│ 33 │ XYZ_PROV   │
│ 34 │ GHRC_DAAC  │
│ 35 │ CSDA       │
│ 36 │ NRSCC      │
│ 37 │ CEOS_EXTRA │
│ 38 │ AMD_KOPRI  │
│ 39 │ AMD_USAPDC │
│ 40 │ LARC       │
│ 41 │ SCIOPS     │
│ 42 │ USGS_EROS  │
│ 43 │ LPCUMULUS  │
│ 44 │ MOPITT     │
│ 45 │ GHRC_CLOUD │
│ 46 │ LPCLOUD    │
│ 47 │ ORNL_DAAC  │
│ 48 │ CCMEO      │
│ 49 │ POCLOUD    │
│ 50 │ PODAAC     │
│ 51 │ SEDAC      │
│ 52 │ GES_DISC   │
│ 53 │ LM_FIRMS   │
│ 54 │ ENVIDAT    │
│ 55 │ OB_CLOUD   │
│ 56 │ USGS       │
└────┴────────────┘

Use LPCLOUD provider to query for available products. This provider is holding the ECOSTRESS L1 and L2 data.

In [4]:
PROVIDER = "LPCLOUD"
catalog = Client.open(f'{STAC_URL}/{PROVIDER}/')
collections = [c for c in catalog.get_children()]

table = rich.table.Table(title="Collections")
table.add_column("ID", style="cyan", no_wrap=True)
table.add_column("Collection")

for c in collections: 
    table.add_row(c.id, c.title)
console.print(table)

                                                    Collections                                                    
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ ID                                         ┃ Collection                                                         ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ASTGTM_NUMNC.v003                          │ ASTER Global Digital Elevation Model Attributes NetCDF V003        │
│ ASTGTM_NC.v003                             │ ASTER Global Digital Elevation Model NetCDF V003                   │
│ ASTGTM.v003                                │ ASTER Global Digital Elevation Model V003                          │
│ WaterBalance_Daily_Historical_GRIDMET.v1.5 │ Daily Historical Water Balance Products for the CONUS              │
│ ECO_L2G_CLOUD.v002                         │ ECOSTRESS Gridded Cloud Mask Instantaneous L2 Global 70 m V002     │
│ ECO_L2G_LSTE.v002                          │ ECOSTRESS Gridded Land Surface Temperature and Emissivity          │
│                                            │ Instantaneous L2 Global 70 m V002                                  │
│ ECO_L1CG_RAD.v002                          │ ECOSTRESS Gridded Top of Atmosphere Calibrated Radiance            │
│                                            │ Instantaneous L1C Global 70 m V002                                 │
│ ECO_L1B_ATT.v002                           │ ECOSTRESS Swath Attitude and Ephemeris Instantaneous L1B Global    │
│                                            │ V002                                                               │
│ ECO_L2_CLOUD.v002                          │ ECOSTRESS Swath Cloud Mask Instantaneous L2 Global 70 m V002       │
│ ECO_L1B_GEO.v002                           │ ECOSTRESS Swath Geolocation Instantaneous L1B Global 70 m V002     │
└────────────────────────────────────────────┴────────────────────────────────────────────────────────────────────┘

## Define AOI and timeframe

In [5]:
time_range = "2023-03-01/2023-03-31"

# GEOJSON can be created on geojson.io
area_of_interest = {
  "coordinates": [
          [
            [
              14.631570827879642,
              48.95580055524857
            ],
            [
              14.631570827879642,
              48.495695436823524
            ],
            [
              15.727627794927486,
              48.495695436823524
            ],
            [
              15.727627794927486,
              48.95580055524857
            ],
            [
              14.631570827879642,
              48.95580055524857
            ]
          ]
        ],
        "type": "Polygon"
}

In [6]:
search = catalog.search(
    collections=["ECO_L2G_LSTE.v002"],
    intersects=area_of_interest,
    datetime=time_range
)
items_eco = search.item_collection()
console.print(f"On {PROVIDER} we found {len(items_eco)} items for the given search query")

On LPCLOUD we found 33 items for the given search query

Get more information about the found items.

In [8]:
table = rich.table.Table(title="Assets in STAC Item")
table.add_column("Asset Key", style="cyan", no_wrap=True)
table.add_column("Description")
for asset_key, asset in items_eco[0].assets.items():
    table.add_row(asset_key, asset.title)

console.print(table)

                                                Assets in STAC Item                                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━┓
┃ Asset Key                                                                                                 ┃ De… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━┩
│ 002/ECOv002_L2G_LSTE_26369_012_20230301T062449_0710_01/ECOv002_L2G_LSTE_26369_012_20230301T062449_0710_01 │ Do… │
│                                                                                                           │ EC… │
│ h5                                                                                                        │ Do… │
│                                                                                                           │ EC… │
│ browse                                                                                                    │ Do… │
│                                                                                                           │ EC… │
│ opendap                                                                                                   │ OP… │
│                                                                                                           │ re… │
│                                                                                                           │ URL │
│                                                                                                           │ (G… │
│                                                                                                           │ DA… │
│                                                                                                           │ :   │
│                                                                                                           │ OP… │
│                                                                                                           │ DA… │
│ metadata                                                                                                  │     │
└───────────────────────────────────────────────────────────────────────────────────────────────────────────┴─────┘

In [9]:
items_eco[0].assets

{'002/ECOv002_L2G_LSTE_26369_012_20230301T062449_0710_01/ECOv002_L2G_LSTE_26369_012_20230301T062449_0710_01': <Asset href=https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/ECO_L2G_LSTE.002/ECOv002_L2G_LSTE_26369_012_20230301T062449_0710_01/ECOv002_L2G_LSTE_26369_012_20230301T062449_0710_01.h5>,
 'h5': <Asset href=https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/ECO_L2G_LSTE.002/ECOv002_L2G_LSTE_26369_012_20230301T062449_0710_01/ECOv002_L2G_LSTE_26369_012_20230301T062449_0710_01.h5.dmrpp>,
 'browse': <Asset href=https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-public/ECO_L2G_LSTE.002/ECOv002_L2G_LSTE_26369_012_20230301T062449_0710_01/ECOv002_L2G_LSTE_26369_012_20230301T062449_0710_01.png>,
 'opendap': <Asset href=https://opendap.earthdata.nasa.gov/collections/C2076113037-LPCLOUD/granules/ECOv002_L2G_LSTE_26369_012_20230301T062449_0710_01>,
 'metadata': <Asset href=https://cmr.earthdata.nasa.gov/search/concepts/G2624435046-LPCLOUD.xml>}

In [10]:
asset_key = f"002/{items_eco[0].id}/{items_eco[0].id}"
items_eco[0].assets[asset_key]

<Asset href=https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/ECO_L2G_LSTE.002/ECOv002_L2G_LSTE_26369_012_20230301T062449_0710_01/ECOv002_L2G_LSTE_26369_012_20230301T062449_0710_01.h5>

## Authenication is done via .netrc file

create .netrc file first based on your Earthdata login

In [10]:
from netrc import netrc
from subprocess import Popen
from platform import system
from getpass import getpass
import os

urs = 'urs.earthdata.nasa.gov'    # Earthdata URL endpoint for authentication
prompts = ['Enter NASA Earthdata Login Username: ',
           'Enter NASA Earthdata Login Password: ']

# Determine the OS (Windows machines usually use an '_netrc' file)
netrc_name = ".netrc" if system()=="Windows" else ".netrc"

# Determine if netrc file exists, and if so, if it includes NASA Earthdata Login Credentials
try:
    netrcDir = os.path.expanduser(f"~/{netrc_name}")
    netrc(netrcDir).authenticators(urs)[0]

# Below, create a netrc file and prompt user for NASA Earthdata Login Username and Password
except FileNotFoundError:
    homeDir = os.path.expanduser("~")
    Popen('touch {0}{2} | echo machine {1} >> {0}{2}'.format(homeDir + os.sep, urs, netrc_name), shell=True)
    Popen('echo login {} >> {}{}'.format(getpass(prompt=prompts[0]), homeDir + os.sep, netrc_name), shell=True)
    Popen('echo \'password {} \'>> {}{}'.format(getpass(prompt=prompts[1]), homeDir + os.sep, netrc_name), shell=True)
    # Set restrictive permissions
    Popen('chmod 0600 {0}{1}'.format(homeDir + os.sep, netrc_name), shell=True)

    # Determine OS and edit netrc file if it exists but is not set up for NASA Earthdata Login
except TypeError:
    homeDir = os.path.expanduser("~")
    Popen('echo machine {1} >> {0}{2}'.format(homeDir + os.sep, urs, netrc_name), shell=True)
    Popen('echo login {} >> {}{}'.format(getpass(prompt=prompts[0]), homeDir + os.sep, netrc_name), shell=True)
    Popen('echo \'password {} \'>> {}{}'.format(getpass(prompt=prompts[1]), homeDir + os.sep, netrc_name), shell=True)

Option 1: Failed to read h5 file with xarray due to groups

In [9]:
import xarray as xr
import fsspec
from netrc import netrc

(username, account, password) = netrc().authenticators("urs.earthdata.nasa.gov")

with fsspec.open(items_eco[0].assets[asset_key].href) as of:
    print(of.info())
    ds = xr.open_dataset(of, group="/HDFEOS/GRIDS/ECO_L2G_LSTE_70m/Data_Fields/EmisWB")
    #ds

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\ebueechi\\.netrc'

Option 2: Tried to read h5 file with datatree - failed

In [37]:
import datatree
import fsspec
from netrc import netrc

(username, account, password) = netrc().authenticators("urs.earthdata.nasa.gov")

with fsspec.open(items_eco[0].assets[asset_key].href) as of:
    ds = datatree.open_datatree(of)
    ds

ValueError: did not find a match in any of xarray's currently installed IO backends ['h5netcdf', 'pydap', 'rasterio', 'zarr']. Consider explicitly selecting one of the installed engines via the ``engine`` parameter, or installing additional IO dependencies, see:
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html
https://docs.xarray.dev/en/stable/user-guide/io.html

Option 3: Read h5 file with native library failed because of fsspec issue

In [12]:
import requests

data = requests.get(items_eco[0].assets[asset_key].href)

# Save file data to local copy
if data.status_code != 200:
    print("ERROR")
with open(f"./{items_eco[0].id}.h5", 'wb') as file:
    file.write(data.content)

In [11]:
import h5py

with h5py.File(f"./{items_eco[0].id}.h5") as f:
    print(f"Keys: {f.keys()}")
    a_group_key = list(f.keys())[0]
    print(a_group_key)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = './ECOv002_L2G_LSTE_26369_012_20230301T062449_0710_01.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)